# Data Preparation

**Project:** *What Makes a Mortgage Cost More?* An app showing homebuyers what borrowers like them paid.

This notebook prepares 2025 HMDA mortgage data for six states:
1. Converts the downloaded files to a faster format
2. Checks the data is complete
3. Loads only the needed columns
4. Keeps only comparable loans
5. Saves the result for analysis (`data/interim/hmda_2025_purchase_filtered.parquet`)

Data decisions are numbered (D1, D2, D3, D4, D5, D6).

## Step 1 · Setup

**Data:** the *Home Mortgage Disclosure Act* (HMDA) data, published by the CFPB. One row = one mortgage application.

**Year: 2025 only (D2).** Mixing years would mix in big interest-rate changes between years.

**States: two per home-price tier (D5)**, based on Zillow median home values (July 2025; US median $370,523):

- **Expensive:** California ($809k), Massachusetts ($686k)
- **Mid:** Florida ($405k), Texas ($308k)
- **Low:** Michigan ($260k), Ohio ($246k)

**Tier rules:** Expensive = top 5 in the US · Mid = within 20% of the US median ($370,523) · Low = 25%+ below it.

*Limitation:* both mid-tier states are in the South and both low-tier states are in the Midwest, so tier differences may partly reflect region.

**Folders:** `data/raw` holds the converted files; `data/interim` holds the filtered result.

In [1]:
from pathlib import Path
import pandas as pd

YEAR = 2025
STATES = ["CA", "MA", "FL", "TX", "MI", "OH"]
STATE_TIERS = {"CA": "Expensive", "MA": "Expensive", "FL": "Mid",
               "TX": "Mid", "MI": "Low", "OH": "Low"}

# Folders: this notebook sits in the project folder
PROJECT_DIR = Path(r"C:\Users\havil\CMSE_midterm_HMDA_project")
RAW_DIR = PROJECT_DIR / "data" / "raw"          # parquet copies of the CSVs
INTERIM_DIR = PROJECT_DIR / "data" / "interim"  # filtered data
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

Project folder: C:\Users\havil\CMSE_midterm_HMDA_project


## Step 2 · Convert the CSV files to parquet

The data was downloaded from the **HMDA Data Browser** (*Dataset by State* → 2025 → *Loan Purpose: Home purchase*), one CSV per state.

Each CSV is converted once to **parquet**, a compressed format that loads much faster and can load only selected columns.

- **`dtype=str`** reads every column as text, so HMDA codes like `"Exempt"` aren't lost.
- **`keep_default_na=False`** keeps the text `"NA"` instead of turning it into a blank, so it can be counted later.
- States already converted are skipped.

In [2]:
for state in STATES:
    csv_file = PROJECT_DIR / f"loan_purposes_1_state_{state}.csv"
    parquet_file = RAW_DIR / f"hmda_{YEAR}_{state}_purchase.parquet"

    if parquet_file.exists():
        print(state, "already converted")
        continue

    # Read every column as text so codes like "NA" and "Exempt" stay as they are
    state_df = pd.read_csv(csv_file, dtype=str, keep_default_na=False)
    state_df.to_parquet(parquet_file, index=False)
    print(state, "converted:", len(state_df), "rows")

CA already converted
MA already converted
FL already converted
TX already converted
MI already converted
OH already converted


## Step 3 · Check the data is complete

The total must match the HMDA Data Browser: **2,282,462 applications**.

**Why this matters:** the first California and Texas downloads were silently cut off (California had 57,100 of 465,779 rows) but opened without errors. This check caught it, and both were re-downloaded (D8).

In [3]:
EXPECTED_TOTAL = 2_282_462   # HMDA Data Browser: 6 states, 2025, home purchase

total = 0
for state in STATES:
    rows = len(pd.read_parquet(RAW_DIR / f"hmda_{YEAR}_{state}_purchase.parquet", columns=["state_code"]))
    print(f"{state} ({STATE_TIERS[state]}): {rows:,} applications")
    total += rows

print(f"\nTotal downloaded: {total:,}  |  Expected: {EXPECTED_TOTAL:,}")
print("Data is complete" if total == EXPECTED_TOTAL else "Mismatch: re-download the small state")

CA (Expensive): 465,779 applications
MA (Expensive): 105,202 applications
FL (Mid): 561,246 applications
TX (Mid): 734,539 applications
MI (Low): 182,986 applications
OH (Low): 232,710 applications

Total downloaded: 2,282,462  |  Expected: 2,282,462
Data is complete


## Step 4 · Load only the needed columns

Of about 99 columns, only these 15 are loaded (plus 8 used only for filtering in Step 5):

| Group | Columns | Why |
|---|---|---|
| Location & outcome | `state_code`, `action_taken` | State, and whether the loan was made |
| **Pricing** | `interest_rate`, `rate_spread`, `total_loan_costs` | The cost of the mortgage |
| Loan features | `loan_type`, `loan_amount`, `loan_to_value_ratio`, `property_value` | Loan program, size and down payment |
| Borrower | `income`, `debt_to_income_ratio` | Borrower finances |
| Loan structure | `loan_term`, `intro_rate_period`, `interest_only_payment`, `balloon_payment` | Used to keep only 30-year fixed-rate loans |

**Key terms**
- **Rate spread:** how far a loan's APR (interest plus fees and mortgage insurance) is above the market average that week, i.e. how much extra the borrower paid.
- **Loan-to-value (LTV):** loan ÷ home value. 90% LTV = 10% down payment.
- **Intro rate period:** filled in only for adjustable-rate mortgages (ARMs), which are priced differently.

`dtype_backend="pyarrow"` stores text compactly, cutting memory use from 5.58 GB to 0.74 GB.

**Duplicates** are counted but not dropped: HMDA has no applicant ID and rounds loan amounts, so identical rows are likely different real loans.

In [4]:
FILTER_COLS = ["loan_purpose", "lien_status", "occupancy_type", "total_units",
               "construction_method", "business_or_commercial_purpose",
               "reverse_mortgage", "open-end_line_of_credit"]

KEEP_COLS = [
    "state_code", "action_taken", "loan_type",           # where, outcome, loan program
    "interest_rate", "rate_spread", "total_loan_costs",  # the prices we explain
    "loan_amount", "loan_to_value_ratio", "property_value",  # loan size and down payment
    "income", "debt_to_income_ratio",                    # borrower finances
    "loan_term", "intro_rate_period",                    # to keep 30-year fixed loans
    "interest_only_payment", "balloon_payment",          # to exclude non-standard loans
]

# Load each state, then stack them into one table
state_tables = []
for state in STATES:
    file = RAW_DIR / f"hmda_{YEAR}_{state}_purchase.parquet"
    state_tables.append(pd.read_parquet(file, columns=KEEP_COLS + FILTER_COLS, dtype_backend="pyarrow"))
df = pd.concat(state_tables, ignore_index=True)

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")

Loaded 2,282,462 rows x 23 columns


## Step 5 · Keep only comparable loans (D3)

To compare prices fairly, every loan must be the same kind of product.

| Filter | Keep | Why |
|---|---|---|
| Home purchase | `1` | Already filtered at download; re-checked |
| First lien | `1` | Second mortgages cost more |
| Principal residence | `1` | Investor and vacation-home loans cost more |
| 1–4 units | `1`–`4` | Larger buildings are commercial loans |
| Site-built | `1` | Manufactured homes are financed differently |
| Not business / reverse / credit line | `2` or `1111` | Different products |

**Why keep `1111` (Exempt)? (D4)** Small lenders don't have to report some fields. Dropping them would bias the data toward big lenders, so they're kept and flagged in Step 6.

All outcomes (approved, denied, withdrawn) are kept; notebook 02 selects approved loans. The funnel table is saved for the app.

In [5]:
counts = [("Downloaded", len(df))]

df = df[df["loan_purpose"] == "1"];                          counts.append(("Home purchase", len(df)))
df = df[df["lien_status"] == "1"];                           counts.append(("First lien", len(df)))
df = df[df["occupancy_type"] == "1"];                        counts.append(("Principal residence", len(df)))
df = df[df["total_units"].isin(["1", "2", "3", "4"])];       counts.append(("1-4 units", len(df)))
df = df[df["construction_method"] == "1"];                   counts.append(("Site-built", len(df)))

# 2 = No, 1111 = Exempt (kept, see D4)
df = df[df["business_or_commercial_purpose"].isin(["2", "1111"])]; counts.append(("Not business purpose", len(df)))
df = df[df["reverse_mortgage"].isin(["2", "1111"])];               counts.append(("Not reverse mortgage", len(df)))
df = df[df["open-end_line_of_credit"].isin(["2", "1111"])];        counts.append(("Not open-end credit line", len(df)))

funnel = pd.DataFrame(counts, columns=["step", "rows_remaining"])
funnel["removed"] = funnel["rows_remaining"].diff().fillna(0).abs().astype(int)
display(funnel)

# Save the funnel for the app's Data & Preparation page
funnel.to_csv(PROJECT_DIR / "data" / "funnel_scope.csv", index=False)

,step,rows_remaining,removed
0,Downloaded,2282462,0
1,Home purchase,2282462,0
2,First lien,2200715,81747
3,Principal residence,1940913,259802
4,1-4 units,1940898,15
5,Site-built,1760505,180393
6,Not business purpose,1757095,3410
7,Not reverse mortgage,1756317,778
8,Not open-end credit line,1747609,8708


### Reading the funnel

**1,747,609 of 2,282,462 applications (76.6%) are kept.**

- **Principal residence** removes the most (259,802): investor and second-home loans.
- **Site-built** removes 180,393: manufactured homes.
- **First lien** removes 81,747: second mortgages taken at purchase.
- **Home purchase** removes 0, confirming the download filter worked.
- The other filters remove very few (12,911 combined).

## Step 6 · Flag Exempt rows and save

We add a `scope_exempt_flag` column that is `True` when a row passed a filter only because the field
was `Exempt` (code `1111`). This lets us later check whether results change when these rows are
excluded (a *sensitivity check*).

The filter-only columns are then dropped, since every remaining row has the same values for them,
and the result is saved. This file keeps **all application outcomes** and is the starting point for
both the midterm and the final project.

In [6]:
scoped = df.copy()

# True if the row was kept only because a field was "Exempt" (1111)
scoped["scope_exempt_flag"] = ((scoped["business_or_commercial_purpose"] == "1111") |
                               (scoped["reverse_mortgage"] == "1111") |
                               (scoped["open-end_line_of_credit"] == "1111"))

scoped = scoped.drop(columns=FILTER_COLS).reset_index(drop=True)
scoped.to_parquet(INTERIM_DIR / f"hmda_{YEAR}_purchase_filtered.parquet", index=False)

print(f"Saved {len(scoped):,} rows x {scoped.shape[1]} columns")
print(f"Rows flagged as Exempt: {scoped['scope_exempt_flag'].sum():,}")

Saved 1,747,609 rows x 16 columns
Rows flagged as Exempt: 19,965


## Summary

**Result:** 1,747,609 applications × 16 columns, saved to `data/interim/hmda_2025_purchase_filtered.parquet`.

| ID | Decision |
|---|---|
| D2 | 2025 data only |
| D3 | Comparable loans only: first-lien purchases of a primary, site-built home |
| D4 | Small-lender "Exempt" rows kept and flagged (19,965 rows, 1.14%) |
| D5 | Six states, two per home-price tier |
| D8 | Download checked against the source; incomplete CA and TX files re-downloaded |

**Next:** notebook 02 cleans the data and analyses mortgage prices.